# Ordered Logistic Regression Results for Adoption Predictors (FAIR²) Exploration with `mlcroissant`

This notebook provides a step-by-step example for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://mlcroissant.readthedocs.io/) data ecosystem library.

### Dataset Source

The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`


In [ ]:
# Ensure `mlcroissant` and necessary packages are installed
!pip install mlcroissant pandas matplotlib

## 1. Data Loading

Load dataset metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load Dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {getattr(metadata, 'name', '')}\nDescription: {getattr(metadata, 'description', '')}")

## 2. Data Overview

Review available record sets, fields, columns, and their `@id`s.

`mlcroissant` exposes metadata describing the structure of the data. Each element (record set, field, column) has a unique `@id`.

In [ ]:
# List all record sets in the dataset, their @id and fields

def get_metadata_ids(metadata):
    out = dict()
    record_sets = getattr(metadata, 'recordSet', [])
    if not isinstance(record_sets, list):
        record_sets = [record_sets]
    for rs in record_sets:
        rs_id = getattr(rs, '@id', None)
        rs_name = getattr(rs, 'name', None)
        fields = getattr(rs, 'field', [])
        if not isinstance(fields, list):
            fields = [fields]
        field_list = []
        for field in fields:
            field_id = getattr(field, '@id', None)
            field_name = getattr(field, 'name', None)
            data_type = getattr(field, 'dataType', None)
            field_list.append({'@id': field_id, 'name': field_name, 'dataType': data_type})
        out[rs_id] = {'name': rs_name, 'fields': field_list}
    return out

all_record_sets = get_metadata_ids(metadata)

if not all_record_sets or all(list(k is None for k in all_record_sets.keys())):
    print("No record sets found in the dataset metadata. Please check the Croissant schema definition.")
else:
    print("Available Record Sets (by @id):")
    for rs_id, v in all_record_sets.items():
        print(f"  - @id: {rs_id} \n    name: {v['name']}")
        print("    Fields:")
        for f in v['fields']:
            print(f"      - @id: {f['@id']} | name: {f['name']} | type: {f['dataType']}")

## 3. Data Extraction

Load data from a specific record set into a pandas DataFrame for analysis. Use the record set and field `@id`s listed above.

In [ ]:
# Choose record sets to extract records for analysis.
# Replace variables below with actual @ids from your dataset if available.

# Example: Assume only one record set if the dataset exposes a single large table.

# Let's gather all @ids of record sets found above
record_sets = [k for k in all_record_sets.keys() if k is not None]

dataframes = dict()
for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)
    print(f"Loaded {len(dataframes[rs_id])} records for record set: {rs_id}")
    print(f"Columns: {dataframes[rs_id].columns.tolist()}")

if record_sets:
    # Pick first available record set for demonstration
    example_rs_id = record_sets[0]
    print(f"\nPreview of record set {example_rs_id} (top 5 rows):")
    display(dataframes[example_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

Let's apply some EDA steps on a numeric column, using field and record set `@id`s as identifiers wherever possible.

We'll:
- Filter records based on a threshold
- Normalize the numeric field
- Optionally group by another field if applicable

In [ ]:
import numpy as np

# Choose an example record set and numeric field (replace these with actual @ids from your overview)
if record_sets:
    rs_id = record_sets[0]
    df = dataframes[rs_id]
    
    # Find a likely numeric field (search for 'log_likelihood', 'coefficient', etc. by column name)
    numeric_candidates = [c for c in df.columns if any(k in c.lower() for k in ['score','likelihood','coef','value','error','iteration','std','val','p_value'])]

    if not numeric_candidates:
        print("No obvious numeric columns found for EDA.")
    else:
        numeric_field = numeric_candidates[0]
        print(f"Using numeric field for EDA: {numeric_field}")
        # Proceed with numeric EDA
        threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered {len(filtered_df)} records with {numeric_field} > {threshold:.2f}")

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a categorical column if available
        # Exclude the numeric from group candidates
        possible_group_fields = [c for c in df.columns if c != numeric_field and (df[c].dtype == object or df[c].dtype == 'category')]

        if possible_group_fields:
            group_field = possible_group_fields[0]
            print(f"Grouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            display(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
else:
    print("No record sets loaded.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset.

We'll create a histogram of the chosen numeric field, and a boxplot grouped by the categorical field if available.

In [ ]:
import matplotlib.pyplot as plt

if record_sets and numeric_candidates:
    # Histogram of numeric field
    plt.figure(figsize=(8,5))
    df[numeric_field].hist(bins=20, color='skyblue', edgecolor='black')
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()
    
    if possible_group_fields:
        group_field = possible_group_fields[0]
        # Boxplot
        plt.figure(figsize=(8,5))
        df.boxplot(column=numeric_field, by=group_field, grid=False, vert=True)
        plt.title(f'{numeric_field} by {group_field}')
        plt.suptitle('')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No fields found for visualization.")

## 6. Conclusion

This notebook demonstrated how to load, inspect, and analyze a FAIR²-structured dataset using `mlcroissant`.

- We explored the dataset's record sets, fields, and metadata via their `@id` identifiers for reproducibility.
- Data loading and DataFrame creation for analysis were shown per record set.
- Example EDA and visualizations provided a starting point for further analysis.

Refer back to the dataset's metadata or the Croissant schema to identify all available fields and their semantics for deeper insights. For documentation and further capabilities, see the [`mlcroissant` documentation](https://mlcroissant.readthedocs.io/).
